<a href="https://colab.research.google.com/github/sudharsanaDA/lithub/blob/main/nlp_ml_tam_ehrs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# Tamil EHR NER Pipeline (Symptoms, Diagnosis, Medications)
# ==========================================

# Install required libraries
!pip install pandas numpy torch transformers datasets seqeval indic-nlp-library stanza evaluate

# -------------------------------
# Step 1: Imports
# -------------------------------
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from datasets import Dataset, DatasetDict
import evaluate # Corrected import
from indicnlp.tokenize import indic_tokenize

# -------------------------------
# Step 2: Load & Prepare Dataset
# -------------------------------
# Example CSV format:
# text,label
# "மருந்து பரிந்துரை: Paracetamol 500mg", "O O O B-MED I-MED"
# Each token has BIO label
df = pd.read_csv("tamil_ehr_ner_sample.csv")  # columns: text, label
display(df.head())

# Tokenize Tamil text
def preprocess_tamil(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|@\S+", "", text)
    text = re.sub(r"[^a-zA-Z\u0B80-\u0BFF\s]", " ", text)
    tokens = indic_tokenize.trivial_tokenize(text)
    return tokens

df['tokens'] = df['text'].apply(preprocess_tamil)
df['labels'] = df['label'].apply(lambda x: x.split())

# -------------------------------
# Step 3: Train-Test Split
# -------------------------------
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['tokens'].tolist(), df['labels'].tolist(), test_size=0.2, random_state=42
)

# -------------------------------
# Step 4: Tokenizer & Label Mapping
# -------------------------------
model_name = "ai4bharat/indic-bert"  # Supports Tamil
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define the list of possible labels
# Assuming entity types are Symptoms, Diagnosis, Medications
possible_labels = ["O", "B-SYMPTOM", "I-SYMPTOM", "B-DIAGNOSIS", "I-DIAGNOSIS", "B-MEDICATION", "I-MEDICATION"]

label2id = {l: i for i, l in enumerate(possible_labels)}
id2label = {i: l for l, i in label2id.items()}

# -------------------------------
# Step 5: Tokenize & Align Labels
# -------------------------------
def tokenize_and_align_labels(tokens_list, labels_list):
    tokenized_inputs = tokenizer(tokens_list, is_split_into_words=True, truncation=True, padding='max_length', max_length=128)
    all_labels = []
    for i, labels in enumerate(labels_list):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)  # Ignored in loss
            elif word_idx != previous_word_idx:
                # Handle cases where label is not in possible_labels
                if word_idx < len(labels) and labels[word_idx] in label2id:
                    aligned_labels.append(label2id[labels[word_idx]])
                else:
                    # Assign a default label like 'O' or -100 if label is not found
                    aligned_labels.append(label2id['O']) # Assign 'O' label
            else:
                # For subword tokens
                if word_idx < len(labels) and labels[word_idx] in label2id:
                     aligned_labels.append(label2id[labels[word_idx]] if labels[word_idx].startswith("I-") else -100)
                else:
                    aligned_labels.append(label2id['O']) # Assign 'O' label
            previous_word_idx = word_idx
        all_labels.append(aligned_labels)
    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

train_encodings = tokenize_and_align_labels(train_texts, train_labels)
test_encodings = tokenize_and_align_labels(test_texts, test_labels)

train_dataset = Dataset.from_dict(train_encodings)
test_dataset = Dataset.from_dict(test_encodings)

# -------------------------------
# Step 6: Load Metric
# -------------------------------
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    return metric.compute(predictions=true_predictions, references=true_labels)

# -------------------------------
# Step 7: Load Model
# -------------------------------
model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=len(possible_labels), id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True
)

# -------------------------------
# Step 8: Training Arguments
# -------------------------------
training_args = TrainingArguments(
    output_dir="./tamil_ner_model",
    eval_strategy="epoch",  # Corrected argument name
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    save_strategy="epoch"
)

# -------------------------------
# Step 9: Trainer
# -------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# -------------------------------
# Step 10: Train
# -------------------------------
trainer.train()

# -------------------------------
# Step 11: Evaluate
# -------------------------------
results = trainer.evaluate()
print("=== NER Model Results ===")
print(results)

# -------------------------------
# Step 12: Save Model
# -------------------------------
model.save_pretrained("./tamil_ner_model")
tokenizer.save_pretrained("./tamil_ner_model")

,text,label
0,காய்ச்சல் புற்றுநோய் நோய் அஸ்துமா புற்றுநோய் த...,B-SYM B-DIAG O I-DIAG B-DIAG I-SYM
1,மருத்துவமனை Amoxicillin நோய் பரிந்துரை நாள் செ...,O B-MED O O O O O O O B-DIAG O I-SYM B-SYM
2,Paracetamol செய்தி மூட்டு வலி நீரிழிவு செய்தி ...,I-MED O I-DIAG I-DIAG O I-DIAG O
3,சோர்வு நாள் சோர்வு பரிந்துரை ரத்த அழுத்தம் Met...,I-SYM O I-SYM O B-DIAG B-MED O O
4,காய்ச்சல் ரத்த அழுத்தம் மருத்துவமனை Metformin ...,B-SYM I-DIAG O I-MED B-SYM I-MED I-MED


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/ai4bharat/indic-bert.
401 Client Error. (Request ID: Root=1-68f79110-5685963323bb7bdb7eea0d65;fff66cad-9d02-44e5-bd78-e56bbe285300)

Cannot access gated repo for url https://huggingface.co/ai4bharat/indic-bert/resolve/main/config.json.
Access to model ai4bharat/indic-bert is restricted. You must have access to it and be authenticated to access it. Please log in.

After adding your Hugging Face token to Colab secrets as `HF_TOKEN`, you can load it like this:

In [ ]:
# Install required libraries
!pip install pandas numpy torch transformers datasets seqeval indic-nlp-library stanza evaluate